# The Advanced RAG Pipeline: Every Fix Behind a Flag

From 5.20 to 5.24 we fixed one failure at a time: hybrid search for keywords, metadata filters for old editions,
reranking for near-misses, and query transformation for two-part questions. Each fix was measured alone. Here
they go into **one pipeline**, each behind a flag, measured on **10 documents** and a **55-question golden set**,
with the time of every step.

```
question → [decompose] → parts
  each part → [self-query: plan a filter] → [rewrite] → query → [+ multi-query] [+ HyDE]
            → search (vector | bm25 | hybrid) → fuse → [rerank] → top k
  union of the parts' chunks → LLM → answer with citations
```

One notebook, no imports from other files:

| Part | What we do |
|---|---|
| 1 | Index the 10 documents into one table, with `year` and `latest` |
| 2 | The pipeline: every step a function, every technique a flag |
| 3 | Three questions the baseline gets wrong |
| 4 | The final golden set: 55 questions |
| 5 | The ladder: switch the flags on one at a time and measure |
| 6 | Time and tokens at every step, and a LangSmith trace |
| 7 | What we ship |

## Setup

In [1]:
%pip install -q --disable-pip-version-check "ragas==0.4.3" "langchain-community==0.4.1" "langchain-classic==1.0.8" "rank-bm25==0.2.2" "instructor==1.17.0" "ranx==0.3.21" langchain-openai langchain-postgres langchain-text-splitters "psycopg[binary]" pgvector pymupdf tiktoken sentence-transformers langsmith python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging, warnings
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

In [3]:
import asyncio, json, os, re, time
from concurrent.futures import ThreadPoolExecutor
from contextvars import ContextVar
from dataclasses import dataclass, replace
from functools import cache, wraps
from pathlib import Path
from threading import Lock
import pandas as pd
import psycopg
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"   # on only for the one trace in Part 6
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
pd.set_option("display.width", 200, "display.max_colwidth", 80)
for key in ["OPENAI_API_KEY", "SUPABASE_DB_URL", "LANGSMITH_API_KEY"]:
    print(f"{key:18} set: {bool(os.getenv(key))}")

OPENAI_API_KEY     set: True
SUPABASE_DB_URL    set: False
LANGSMITH_API_KEY  set: True


# Part 1: Index the 10 documents

The 7 documents from 5.14 and the 3 older editions from 5.22, in one new table, `advanced_chunks`. The same loaders
and splitter as 5.14, and the same `year` and `latest` columns as 5.22.

## 1. Load the pages

PDFs with PyMuPDF. The scanned bill (5.6) and the Wikipedia page (5.7) were extracted earlier and saved as JSON lines.

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

PDFS = [*sorted(Path("../5.4_pdfs_pymupdf/pdfs").glob("*.pdf")), *sorted(Path("../5.22_metadata_filters/pdfs").glob("*.pdf"))]

EXTRACTED = [Path("../5.6_scanned_pdfs_ocr/extracted/PDF_For_OCR.jsonl"), Path("../5.7_html_web_pages/extracted/rag_wikipedia.jsonl")]

YEARS = {"nist-sp-800-63b-4": 2025, "irs-w9": 2024, "census-income-2023": 2023, "bert": 2019, "prod-kx40-manual": 2024,
         "PDF_For_OCR": 2020, "wikipedia-rag": 2025, "nist-sp-800-63b-3": 2017, "irs-w9-2018": 2018, "census-income-2022": 2022}
         
OLD = {"nist-sp-800-63b-3", "irs-w9-2018", "census-income-2022"}

def load_pages():
    pages = []
    for pdf in PDFS:
        for page in PyMuPDFLoader(str(pdf)).load():
            pages.append(Document(page.page_content, metadata={"doc_id": pdf.stem, "source": pdf.name, "page": page.metadata["page"] + 1}))
    for path in EXTRACTED:
        for line in path.read_text(encoding="utf-8").splitlines():
            saved_page = json.loads(line)
            pages.append(Document(saved_page["page_content"], metadata={
                "doc_id": saved_page["metadata"]["doc_id"], "source": path.name, "page": saved_page["metadata"].get("page", 0) + 1}))
    return pages

pages = load_pages()
pd.Series({doc_id: sum(p.metadata["doc_id"] == doc_id for p in pages) for doc_id in YEARS}, name="pages").to_frame().T

FileNotFoundError: [Errno 2] No such file or directory: '../5.6_scanned_pdfs_ocr/extracted/PDF_For_OCR.jsonl'

## 2. Chunk

500 tokens with 100 overlap, as in 5.14. Each chunk gets an id (`doc-p<page>-c<index>`), its token count, its edition
`year`, and `latest` (False only for the three older editions).

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", separators=["\n\n", ". ", "\n", " ", ""], chunk_size=500, chunk_overlap=100,
    keep_separator="end")
    
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_pages(pages):
    chunks, seen = splitter.split_documents(pages), {}
    for c in chunks:
        m = c.metadata
        m["chunk_index"] = seen[m["doc_id"]] = seen.get(m["doc_id"], -1) + 1
        m["n_tokens"] = len(tokenizer.encode(c.page_content))
        m["year"], m["latest"] = YEARS[m["doc_id"]], m["doc_id"] not in OLD
        c.id = f"{m['doc_id']}-p{m['page']}-c{m['chunk_index']}"
    return chunks

chunks = chunk_pages(pages)
by_id = {c.id: c for c in chunks}
print(len(chunks), "chunks")

## 3. Embed and store

`build_table` runs once: it embeds the 1,134 chunks (about 450,000 tokens, about $0.01) and writes them to Supabase.
After that the table exists and this cell only counts the rows.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from pgvector.psycopg import register_vector

TABLE = "advanced_chunks"
embedder = OpenAIEmbeddings(model="text-embedding-3-small")

def build_table(conn, chunks):
    vectors = embedder.embed_documents([c.page_content for c in chunks])
    conn.execute(f"""CREATE TABLE {TABLE} (chunk_id text PRIMARY KEY, doc_id text, source text, page int,
        chunk_index int, n_tokens int, content text, embedding vector(1536), year int, latest boolean)""")
    conn.execute(f"ALTER TABLE {TABLE} ENABLE ROW LEVEL SECURITY")
    rows = [[c.id, m["doc_id"], m["source"], m["page"], m["chunk_index"], m["n_tokens"], c.page_content, v, m["year"], m["latest"]]
            for c, v in zip(chunks, vectors) for m in [c.metadata]]
    conn.cursor().executemany(f"INSERT INTO {TABLE} VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)", rows)

with psycopg.connect(os.getenv("SUPABASE_DB_URL"), autocommit=True) as conn:
    register_vector(conn)
    if not conn.execute("SELECT to_regclass(%s)", (TABLE,)).fetchone()[0]:
        build_table(conn, chunks)
    editions = pd.DataFrame(conn.execute(f"SELECT doc_id, year, latest, count(*) FROM {TABLE} GROUP BY 1, 2, 3 ORDER BY 1").fetchall(),
                            columns=["doc_id", "year", "latest", "chunks"])
editions

1,134 chunks from 10 documents, and 498 of them (44%) are from the three older editions. Nearly every NIST, W-9 or
Census question now has a near-twin in an older edition that a search can pick by mistake.

# Part 2: The pipeline

## 4. Every step is a function, and `@step` times it

Each step of the pipeline is a small function. `@step` wraps every one of them and does three things:

- **Saves** the result in `experiments/steps.json`, with the seconds and tokens it took. A step runs once per
  input; a rerun of the notebook reads it back and shows the same queries, answers and times.
- **Traces**: every call adds a line to the current question's trace (step, input, seconds, tokens).
- **`@traceable`**: the same functions show up as a tree in LangSmith when tracing is on (Part 6).

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback
from langsmith import traceable

STEPS = Path("experiments/steps.json")
saved = json.loads(STEPS.read_text()) if STEPS.exists() else {}
current_trace = ContextVar("current_trace", default=None)

def step(name, lock=None):
    def wrap(fn):
        memo = saved.setdefault(name, {})
        @wraps(fn)
        def run(*args):
            key = json.dumps(args)
            if key not in memo:
                with lock or Lock():
                    start = time.perf_counter()
                    with get_usage_metadata_callback() as usage:
                        result = fn(*args)
                    memo[key] = {"result": result, "seconds": round(time.perf_counter() - start, 3),
                                 "tokens": sum(u.get("total_tokens", 0) for u in usage.usage_metadata.values())}
            hit, trace = memo[key], current_trace.get()
            if trace is not None:
                trace.append({"step": name, "input": " ".join(str(args[0]).split())[:70],
                              "seconds": hit["seconds"], "tokens": hit["tokens"]})
            return hit["result"]
        return traceable(name=name)(run)
    return wrap

def save():
    STEPS.parent.mkdir(exist_ok=True)
    STEPS.write_text(json.dumps(saved, ensure_ascii=False))

## 5. Search: vector, BM25, hybrid, with an optional filter

A filter is `None` (search everything), `("latest", True)` or `("year", 2017)`. BM25 keeps one index per filter.
Hybrid fuses the two lists with reciprocal rank fusion (`fuse`): each list adds `1 / (60 + rank)` to a chunk's
score. That's what `EnsembleRetriever` did in 5.21.1 with equal weights. Searches return chunk ids.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_postgres import PGEngine, PGVectorStore

LATEST = ("latest", True)
engine = PGEngine.from_connection_string(url=os.getenv("SUPABASE_DB_URL").replace("postgresql://", "postgresql+psycopg://"))
store = PGVectorStore.create_sync(engine=engine, table_name=TABLE, embedding_service=embedder,
                                  id_column="chunk_id", metadata_columns=["doc_id", "page", "year", "latest"])

def passes(chunk, filter):
    return filter is None or chunk.metadata[filter[0]] == filter[1]

def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

@cache
def bm25_index(filter):
    return BM25Retriever.from_documents([c for c in chunks if passes(c, filter)], preprocess_func=tokenize)

@step("vector")
def vector_search(query, k, filter):
    return [d.id for d in store.similarity_search(query, k=k, filter=dict([filter]) if filter else None)]

@step("bm25")
def bm25_search(query, k, filter):
    return [d.id for d in bm25_index(filter).model_copy(update={"k": k}).invoke(query)]

def fuse(lists, k):
    scores = {}
    for ids in lists:
        for rank, id in enumerate(ids, 1):
            scores[id] = scores.get(id, 0) + 1 / (60 + rank)
    return sorted(scores, key=scores.get, reverse=True)[:k]

def union(lists):
    return list(dict.fromkeys(id for ids in lists for id in ids))

def search(query, retriever, k, filter=None):
    if retriever == "vector":
        return vector_search(query, k, filter)
    if retriever == "bm25":
        return bm25_search(query, k, filter)
    return fuse([vector_search(query, k, filter), bm25_search(query, k, filter)], k)

## 6. The LLM steps before search

The prompts from 5.22 (the search plan) and 5.24.1 (rewrite, multi-query, decomposition, HyDE), each its own step.

**Self-query** (5.22): the planner reads the list of editions and fills in a `year` only when the question asks
for one. `to_filter` turns that into a filter: no year → latest editions only; a year we have → that edition; a year
we don't have → no filter.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

chat = ChatOpenAI(model="gpt-5.6-luna", reasoning_effort="none")

CATALOG = "\n".join(f"- {r.doc_id}: {r.year} edition" + (" (latest)" if r.latest else "") for r in editions.itertuples())
EDITION_YEARS = set(editions.year)

class SearchPlan(BaseModel):
    year: int | None = Field(None, description="The edition year, the year in a document's title. Set it only when "
                                               "the question asks about a specific edition, version or year. "
                                               "Otherwise null.")

planner = ChatPromptTemplate.from_messages([
    ("system", "You plan searches over a document store. It holds these documents:\n{catalog}\n\n"
               "Fill in the search plan for the user's question."),
    ("human", "{question}")]) | chat.with_structured_output(SearchPlan)

@step("plan")
def plan_year(question):
    return planner.invoke({"catalog": CATALOG, "question": question}).year

def to_filter(year):
    if year is None:
        return LATEST
    if year in EDITION_YEARS:
        return ("year", year)
    return None

In [ ]:
def instruction(text):
    return ChatPromptTemplate.from_messages([("system", text), ("human", "{question}")]) | chat

rewriter = instruction("Rewrite this question as one short search query for a document search engine. Reply with the query only.")
multi_writer = instruction("Write 3 different versions of this question for a document search engine, one per line. "
                           "Reply with the questions only.")
decomposer = instruction("Split this question into separate standalone questions, one per line. If it asks only one thing, "
                         "return it unchanged. Reply with the questions only.")
hyde_writer = instruction("Write a short passage, as it might appear in a document, that answers this question. "
                          "Reply with the passage only.")

def lines(text):
    return [re.sub(r"^\s*(?:\d+[.)]|[-•*])\s*", "", line).strip() for line in text.splitlines() if line.strip()]

@step("decompose")
def split(question):
    return lines(decomposer.invoke({"question": question}).content) or [question]

@step("rewrite")
def rewrite(question):
    return rewriter.invoke({"question": question}).content

@step("multi_query")
def variants(question):
    return lines(multi_writer.invoke({"question": question}).content)[:3]

@step("hyde")
def fake_answer(question):
    return hyde_writer.invoke({"question": question}).content

## 7. Rerank and answer

The rerankers from 5.23.1, loaded from the Hugging Face cache; bge is loaded here, so its loading time isn't counted
as a rerank. They run one at a time (`lock`), so the time of a rerank is the rerank alone. The answer prompt is 5.22's: numbered sources with their edition, `[n]`
citations, and a fixed refusal.

In [ ]:
from sentence_transformers import CrossEncoder

RERANKERS = {"bge": "BAAI/bge-reranker-v2-m3", "minilm": "cross-encoder/ms-marco-MiniLM-L6-v2"}

@cache
def cross_encoder(name):
    return CrossEncoder(RERANKERS[name])

@step("rerank", lock=Lock())
def rerank(question, ids, model, k):
    hits = cross_encoder(model).rank(question, [by_id[id].page_content for id in ids])
    return [ids[hit["corpus_id"]] for hit in hits[:k]]

print("bge on", cross_encoder("bge").device)

REFUSAL = "I don't know based on the documents."
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the numbered sources. After each sentence, cite the sources it uses, "
               "like [1] or [2][3]. If the sources do not contain the answer, reply exactly: " + REFUSAL),
    ("human", "Sources:\n\n{context}\n\nQuestion: {question}")])

@step("generate")
def generate(question, ids):
    sources = "\n\n".join(f"[{i}] {by_id[id].metadata['doc_id']} ({by_id[id].metadata['year']} edition), "
                          f"page {by_id[id].metadata['page']}\n{by_id[id].page_content}" for i, id in enumerate(ids, 1))
    return (answer_prompt | chat).invoke({"context": sources, "question": question}).content

## 8. `Config`: one flag per technique

A `Config` says which steps run. It's frozen, so a config can't change by accident; `replace` makes a new one.

In [ ]:
@dataclass(frozen=True)
class Config:
    name: str
    retriever: str = "vector"           # vector | bm25 | hybrid
    self_query: bool = False            # plan a year filter per part
    query_steps: tuple[str, ...] = ()   # any of: decompose, rewrite, multi_query, hyde
    rerank: str | None = None           # None | bge | minilm
    rerank_depth: int = 20              # candidates the reranker reads
    k: int = 4                          # chunks per part sent to the LLM

## 9. `retrieve` and `ask`

The order, for every config:

1. **Decompose** (if on): split the question into parts. Otherwise there's one part, the question.
2. For each part:
   - **Self-query** (if on): plan a filter.
   - **Rewrite** (if on): the part becomes one search query.
   - **Multi-query** adds 3 versions of the query; **HyDE** adds a fake answer.
   - Search each query, fuse the lists, then **rerank** the top `rerank_depth` down to `k` (if on), or keep the
     top `k`. Reranking scores the chunks against the part, not the rewritten query.
3. Union the parts' chunks (a two-part question reads up to 8), and generate.

`ask` returns the answer, the chunk ids and the trace.

In [ ]:
def retrieve(question, config):
    parts = split(question) if "decompose" in config.query_steps else [question]
    found = []
    for part in parts:
        filter = to_filter(plan_year(part)) if config.self_query else None
        query = rewrite(part) if "rewrite" in config.query_steps else part
        queries = [query]
        if "multi_query" in config.query_steps:
            queries += variants(query)
        if "hyde" in config.query_steps:
            queries.append(fake_answer(query))
        depth = config.rerank_depth if config.rerank else config.k
        candidates = fuse([search(q, config.retriever, depth, filter) for q in queries], depth)
        found.append(rerank(part, candidates, config.rerank, config.k) if config.rerank else candidates[:config.k])
    return union(found)

@traceable(name="ask")
def ask(question, config):
    trace = []
    token = current_trace.set(trace)
    try:
        ids = retrieve(question, config)
        answer = generate(question, ids)
    finally:
        current_trace.reset(token)
    return {"answer": answer, "ids": ids, "trace": trace}

## 10. The six configs

The baseline is plain vector search over all 10 documents. Each rung switches one more flag on; the last one
switches everything on.

In [15]:
BASELINE = Config("baseline")
HYBRID = replace(BASELINE, name="+hybrid", retriever="hybrid")
SELF_QUERY = replace(HYBRID, name="+self-query", self_query=True)
DECOMPOSE = replace(SELF_QUERY, name="+decomposition", query_steps=("decompose",))
RERANK = replace(DECOMPOSE, name="+reranker", rerank="bge")
EVERYTHING = replace(RERANK, name="everything", query_steps=("decompose", "rewrite", "multi_query", "hyde"))
LADDER = [BASELINE, HYBRID, SELF_QUERY, DECOMPOSE, RERANK, EVERYTHING]
pd.DataFrame([vars(c) for c in LADDER]).set_index("name")

,retriever,self_query,query_steps,rerank,rerank_depth,k
name,,,,,,
baseline,vector,False,(),NaN,20,4
+hybrid,hybrid,False,(),NaN,20,4
+self-query,hybrid,True,(),NaN,20,4
+decomposition,hybrid,True,"(decompose,)",NaN,20,4
+reranker,hybrid,True,"(decompose,)",bge,20,4
everything,hybrid,True,"(decompose, rewrite, multi_query, hyde)",bge,20,4


# Part 3: Three questions the baseline gets wrong

First the golden set (Part 4 looks at it closely), and `show`: the question, the trace with the time of every
step, the chunks (✅: holds the evidence, from the right edition), the answer and the expected answer.

In [16]:
SEP = "||"
golden = pd.read_csv("datasets/golden_final.csv", keep_default_na=False)
q = golden.set_index("id")

def pieces(row):
    return list(zip(row.evidence.split(SEP), row.needs.split(SEP))) if row.evidence else []

def holds(piece, chunk):
    evidence, needs = piece
    edition = chunk.metadata["latest"] if needs == "latest" else chunk.metadata["year"] == int(needs)
    return edition and " ".join(evidence.lower().split()) in " ".join(chunk.page_content.lower().split())

def show(qid, config):
    row, out = q.loc[qid], ask(q.loc[qid].question, config)
    print(f"{qid} ({row.type}, needs {row.needs}): {row.question}\nconfig: {config.name}\n")
    for t in out["trace"]:
        print(f"   {t['step']:11} {t['seconds']:5.2f} s {t['tokens']:5} tokens   {t['input']}")
    print(f"   {'total':11} {sum(t['seconds'] for t in out['trace']):5.2f} s\n")
    for rank, id in enumerate(out["ids"], 1):
        c = by_id[id]
        mark = "✅" if any(holds(p, c) for p in pieces(row)) else "  "
        edition = f"{c.metadata['year']}{'' if c.metadata['latest'] else ' old'}"
        print(f"{mark} {rank}. {id:30} {edition:9} {' '.join(c.page_content.split())[:60]}")
    print(f"\nLLM:      {out['answer']}\nExpected: {row.reference_answer}")

## 11. A two-part question

In [17]:
show("t02", BASELINE)

t02 (two-part, needs latest||latest): What is the minimum length for a password, and when was the term RAG introduced?
config: baseline

   vector       1.61 s     0 tokens   What is the minimum length for a password, and when was the term RAG i
   generate     1.62 s  1776 tokens   What is the minimum length for a password, and when was the term RAG i
   total        3.24 s

   1. nist-sp-800-63b-4-p98-c169     2025      These attacks are outside of the scope of this Appendix. A.2
   2. nist-sp-800-63b-3-p79-c134     2017 old  NIST SP 800-63B DIGITAL IDENTITY GUIDELINES: AUTHENTICATION 
   3. nist-sp-800-63b-3-p78-c132     2017 old  NIST SP 800-63B DIGITAL IDENTITY GUIDELINES: AUTHENTICATION 
   4. nist-sp-800-63b-4-p98-c168     2025      NIST SP 800-63B-4 July 2025 Digital Identity Guidelines Auth

LLM:      I don't know based on the documents.
Expected: A password used on its own must be at least 15 characters long. A password that is only one part of multi-factor authentication may

In [18]:
show("t02", EVERYTHING)

t02 (two-part, needs latest||latest): What is the minimum length for a password, and when was the term RAG introduced?
config: everything

   decompose    1.36 s    77 tokens   What is the minimum length for a password, and when was the term RAG i
   plan         0.91 s   269 tokens   What is the minimum length for a password?
   rewrite      1.12 s    46 tokens   What is the minimum length for a password?
   multi_query  1.21 s    66 tokens   minimum password length requirement
   hyde         1.08 s    61 tokens   minimum password length requirement
   vector       0.58 s     0 tokens   minimum password length requirement
   bm25         0.03 s     0 tokens   minimum password length requirement
   vector       0.39 s     0 tokens   What is the minimum required password length?
   bm25         0.00 s     0 tokens   What is the minimum required password length?
   vector       0.40 s     0 tokens   What is the shortest password allowed?
   bm25         0.00 s     0 tokens   What is the

- **Baseline:** one search, four chunks, all about password length (two of them from the 2017 edition), and none with
  either rule. The bot refuses the whole question, even the password half.
- **Everything on:** the question is split in two, and each part is planned, rewritten, expanded into 5 queries,
  searched and reranked. Of the 8 chunks, the 15-character rule is 1st and the RAG history 5th and 6th. Both halves
  are answered and cited.
- **The price:** 21 seconds instead of 3, and a trace of 32 lines instead of 2.

## 12. The wrong edition

In [19]:
show("v01", BASELINE)

v01 (version, needs latest): What was the real median household income?
config: baseline

   vector       0.33 s     0 tokens   What was the real median household income?
   generate     0.97 s  1913 tokens   What was the real median household income?
   total        1.30 s

   1. census-income-2022-p8-c24      2022 old  2 Income in the United States: 2022 U.S. Census Bureau HOUSE
   2. census-income-2022-p8-c25      2022 old  Among family house­ holds, married couples had the highest m
   3. census-income-2022-p10-c30     2022 old  citizens and those who are not U.S. citizens. Neither group 
✅ 4. census-income-2023-p7-c22      2023      To further protect respondent privacy, all estimates in this

LLM:      The real median household income was $74,580 in 2022. [1]
Expected: Real median household income was $80,610 in 2023.


In [20]:
show("v01", EVERYTHING)

v01 (version, needs latest): What was the real median household income?
config: everything

   decompose    1.03 s    58 tokens   What was the real median household income?
   plan         0.97 s   268 tokens   What was the real median household income?
   rewrite      1.09 s    45 tokens   What was the real median household income?
   multi_query  0.99 s    68 tokens   real median household income
   hyde         1.25 s    91 tokens   real median household income
   vector       0.88 s     0 tokens   real median household income
   bm25         0.00 s     0 tokens   real median household income
   vector       0.82 s     0 tokens   What is the real median household income?
   bm25         0.00 s     0 tokens   What is the real median household income?
   vector       0.53 s     0 tokens   How has real median household income changed over time?
   bm25         0.00 s     0 tokens   How has real median household income changed over time?
   vector       0.60 s     0 tokens   What is the

- **Baseline:** the top 3 chunks are from the **2022** report, and the bot answers *"$74,580 in 2022"*, with a
  citation. It's wrong but sounds confident, and nothing in the answer says it's from an old edition.
- **Everything on:** the planner finds no year in the question, so the filter keeps the latest editions only. All
  four chunks are from 2023, and the answer is right: $80,610.

## 13. The password question from 5.23

In [21]:
show("n01", BASELINE)

n01 (near-miss, needs latest): Is an 8-character password long enough?
config: baseline

   vector       0.46 s     0 tokens   Is an 8-character password long enough?
   generate     0.77 s  1526 tokens   Is an 8-character password long enough?
   total        1.23 s

   1. nist-sp-800-63b-3-p79-c134     2017 old  NIST SP 800-63B DIGITAL IDENTITY GUIDELINES: AUTHENTICATION 
   2. nist-sp-800-63b-3-p78-c133     2017 old  Keystroke logging, phishing, and social engineering attacks 
   3. nist-sp-800-63b-4-p99-c170     2025      NIST SP 800-63B-4 July 2025 Digital Identity Guidelines Auth
   4. nist-sp-800-63b-4-p98-c169     2025      These attacks are outside of the scope of this Appendix. A.2

LLM:      I don't know based on the documents.
Expected: No. A password used on its own as single-factor authentication must be at least 15 characters long.


In [22]:
show("n01", EVERYTHING)

n01 (near-miss, needs latest): Is an 8-character password long enough?
config: everything

   decompose    0.93 s    60 tokens   Is an 8-character password long enough?
   plan         0.68 s   269 tokens   Is an 8-character password long enough?
   rewrite      0.82 s    48 tokens   Is an 8-character password long enough?
   multi_query  1.00 s    73 tokens   8-character password security adequacy
   hyde         1.79 s   107 tokens   8-character password security adequacy
   vector       0.67 s     0 tokens   8-character password security adequacy
   bm25         0.00 s     0 tokens   8-character password security adequacy
   vector       0.78 s     0 tokens   How secure is an 8-character password?
   bm25         0.00 s     0 tokens   How secure is an 8-character password?
   vector       0.62 s     0 tokens   Are 8-character passwords sufficiently secure?
   bm25         0.00 s     0 tokens   Are 8-character passwords sufficiently secure?
   vector       0.79 s     0 tokens   What 

- **Baseline:** four chunks about password length, two from 2017 and two from the 2025 appendix, and none with the
  15-character rule. The bot refuses.
- **Everything on:** the rule is 1st, and the answer is better than the answer key: 8 characters is the minimum
  only as part of multi-factor authentication; on its own a password needs 15.

In [23]:
save()

# Part 4: The final golden set

55 questions. The 27 we've used since 5.22 (`seen`), and 28 new ones, one group per failure the fixes target. Every
evidence phrase was checked against the chunks of the edition in `needs`. A two-part question has two pieces of
evidence and two `needs`, joined with `||`. `changelog` records every edit made to a question after the first run.

In [24]:
golden.groupby(["group", "type"]).size().rename("questions").to_frame().T

group         new                                                          seen                                   
type      edition keyword near-miss paraphrase table two-part unanswerable fact table unanswerable version wording
questions       4       4         4          3     2        8            3   13     4            2       6       2

In [25]:
golden[golden.group == "new"][["id", "type", "question", "needs"]]

,id,type,question,needs
27,t01,two-part,"Which pre-training task did BERT add next to the masked language model, and ...",latest||latest
28,t02,two-part,"What is the minimum length for a password, and when was the term RAG introdu...",latest||latest
29,t03,two-part,"What was the median household income in the United States in 2023, and who h...",latest||latest
30,t04,two-part,"Do users have to change their passwords regularly, and where does RAG store ...",latest||latest
31,t05,two-part,"Does replacing words with random ones hurt BERT's training, and what should ...",latest||latest
32,t06,two-part,"How much income put a household in the top 5 percent in 2023, and can a fake...",latest||latest
33,t07,two-part,"How much was the service charge on The Buck Stops Here bill, and can I use S...",latest||latest
34,t08,two-part,"When should an entity check box 3b on Form W-9, and what was the grand total...",latest||latest
35,e01,edition,"Under the 2017 NIST guideline, how long did a subscriber-chosen memorized se...",2017
36,e02,edition,"According to the 2022 income report, what was the real median household income?",2022


55 questions: 27 we've used since 5.22 and 28 new, one group per failure:

- **two-part (8):** two golden questions from different documents, joined with *", and"*.
- **edition (4):** the answer is in an older edition, and the newer one says something else.
- **keyword (4):** codes and ids (E-417, a GSTIN, FATCA code B, BERT-LARGE's sizes).
- **near-miss (4):** many chunks on the topic, one with the answer.
- **paraphrase (3):** none of the chunk's words in the question.
- **table (2)** and **refusal (3)**. *"BERT on ImageNet"* is a trap: ImageNet appears once, in BERT's references.

The edits after the first run, each with its reason:

In [26]:
golden[golden.changelog != ""][["id", "changelog"]]

,id,changelog
6,q07,"reference widened: the answer key named only individuals, the form's definit..."
16,q17,"reference names exempt payee 7, which correct answers add from the same table"
17,q18,reference trimmed to the rule itself; the long version marked down correct a...
26,v06,"both reports give a 2021 median ($76,330 in 2022 dollars, $79,260 in 2023 do..."
29,t03,"q07: reference widened: the answer key named only individuals, the form's de..."
34,t08,q18: reference trimmed to the rule itself; the long version marked down corr...
50,b01,"dropped the 2.5% rate: the scanned bill reads 'CGST82.50s', so no answer can..."


Five edits after the first run, and two two-part rows inherit theirs:

- **q07, q17, q18 and b01** fix the answer key, not the question. Each time, correct answers were marked down
  because the reference was narrower than the document, or held a detail no answer could confirm.
- **v06** was ambiguous: both reports give a 2021 median, in different dollars. It now names the 2022 report.

Two rows are left failing on purpose: **q05** (a 2010 value inside a table row; no config finds it) and **q03**
(SMS: the extra steps make it worse).

# Part 5: The ladder

## 14. Run every config on every question

`run` asks each question with a config, 4 at a time, and keeps the answers in `experiments/answers.jsonl`, keyed by
the config and the question text: an edited question runs again, the others don't.

In [27]:
ANSWERS = Path("experiments/answers.jsonl")
done = {}
if ANSWERS.exists():
    for line in ANSWERS.read_text().splitlines():
        record = json.loads(line)
        done[record["key"]] = record

def key(config, row):
    return f"{config}|{row.id}|{row.question}"

def run(config, rows=golden):
    todo = [row for row in rows.itertuples() if key(config, row) not in done]
    with ThreadPoolExecutor(4) as pool:
        outs = list(pool.map(lambda row: ask(row.question, config), todo))
    with ANSWERS.open("a") as f:
        for row, out in zip(todo, outs):
            done[key(config, row)] = record = {"key": key(config, row), "config": config.name, "id": row.id, **out}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    save()
    return [done[key(config, row)] for row in rows.itertuples()]

ANSWERS.parent.mkdir(exist_ok=True)
outputs = {}
for config in LADDER:
    start = time.perf_counter()
    outputs[config.name] = run(config)
    print(f"{config.name:15} {time.perf_counter() - start:6.1f} s")

baseline           0.0 s
+hybrid            0.0 s
+self-query        0.0 s
+decomposition     0.0 s
+reranker          0.0 s
everything         0.0 s


## 15. Retrieval, with ranx

Scored on exactly the chunks sent to the LLM (4 per part, so up to 8 for a two-part question), with no cutoff. Each
piece of evidence is one relevant item (`e0`, `e1`), and a chunk counts only if it's from the right edition.
`complete`: the share of questions with **every** piece of evidence found.

In [28]:
from ranx import Qrels, Run, evaluate

METRICS = ["hit_rate", "recall", "mrr", "ndcg"]
answerable = golden[~golden.should_refuse]
qrels = Qrels({row.id: {f"e{i}": 1 for i in range(len(pieces(row)))} for row in answerable.itertuples()})

def ranked(row, ids):
    run = {}
    for rank, id in enumerate(ids, 1):
        hit = next((f"e{i}" for i, p in enumerate(pieces(row)) if f"e{i}" not in run and holds(p, by_id[id])), id)
        run.setdefault(hit, 1 / rank)
    return run

def found_all(row, ids):
    return all(any(holds(p, by_id[id]) for id in ids) for p in pieces(row))

def retrieval_scores(outs, rows=answerable):
    ids = {out["id"]: out["ids"] for out in outs}
    scores = evaluate(Qrels({r.id: qrels.qrels[r.id] for r in rows.itertuples()}),
                      Run({r.id: ranked(r, ids[r.id]) for r in rows.itertuples()}), METRICS)
    return {**{m: float(scores[m]) for m in METRICS},
            "complete": sum(found_all(r, ids[r.id]) for r in rows.itertuples()) / len(rows)}

retrieval = pd.DataFrame({name: retrieval_scores(outs) for name, outs in outputs.items()}).T.round(2)
retrieval

,hit_rate,recall,mrr,ndcg,complete
baseline,0.84,0.77,0.69,0.67,0.70
+hybrid,0.88,0.81,0.71,0.71,0.74
+self-query,0.92,0.85,0.77,0.77,0.78
+decomposition,0.92,0.92,0.76,0.79,0.92
+reranker,0.96,0.95,0.86,0.86,0.94
everything,0.96,0.95,0.88,0.88,0.94


Every column goes up from the baseline to everything on: hit rate 0.84 → 0.96, MRR 0.69 → 0.88, and `complete`
(every piece of evidence found) 0.70 → 0.94. Each rung does its own job:

- **Hybrid** and **self-query** lift hit rate (0.84 → 0.88 → 0.92).
- **Decomposition** gives the biggest jump in `complete`: 0.78 → 0.92.
- **The reranker** moves the evidence up: MRR 0.76 → 0.86.
- **Rewrite, multi-query and HyDE together** add nothing to `complete` and 0.02 to MRR.

`complete` by question type:

In [29]:
complete_by_type = pd.DataFrame({name: {t: retrieval_scores(outs, rows)["complete"]
                                        for t, rows in answerable.groupby("type")}
                                 for name, outs in outputs.items()}).round(2)
complete_by_type

,baseline,+hybrid,+self-query,+decomposition,+reranker,everything
edition,0.75,0.75,0.75,0.75,1.00,1.00
fact,0.85,0.85,1.00,1.00,1.00,1.00
keyword,1.00,1.00,1.00,1.00,1.00,1.00
near-miss,0.75,0.75,0.75,0.75,1.00,1.00
paraphrase,1.00,1.00,1.00,1.00,1.00,1.00
table,0.67,0.67,0.67,0.67,0.83,0.83
two-part,0.00,0.12,0.12,1.00,0.88,0.88
version,0.83,1.00,1.00,1.00,1.00,1.00
wording,1.00,1.00,1.00,1.00,0.50,0.50


- **Two-part:** 0.00 → 0.12 → **1.00** at decomposition. No search fixes a question that asks two things.
- **Edition and near-miss:** only the reranker fixes them (0.75 → 1.00). It reads the question and the chunk
  together, and can tell 2017's rule from 2025's.
- **The cost of reranking:** one two-part question loses a half, and wording drops from 1.00 to 0.50 because of
  q03 (SMS). The reranker keeps four other pages on SMS and out-of-band codes, and drops the PSTN rule that answers
  it.

## 16. Answers, with Ragas

The judge from 5.20.2 scores every answer; the scores go into the same `answers.jsonl` record, with the reference
they were scored against. A rerun pays only for new answers and edited references.

In [30]:
from openai import AsyncOpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.metrics.collections import (AnswerRelevancy, ContextPrecision, ContextRecall, FactualCorrectness,
                                       Faithfulness)

client = AsyncOpenAI()
judge = llm_factory("gpt-5.6-luna", client=client, reasoning_effort="none", max_tokens=4096)
judge.model_args["max_completion_tokens"] = judge.model_args.pop("max_tokens")
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)
faithfulness, factual_correctness = Faithfulness(llm=judge), FactualCorrectness(llm=judge)
answer_relevancy = AnswerRelevancy(llm=judge, embeddings=embeddings)
context_precision, context_recall = ContextPrecision(llm=judge), ContextRecall(llm=judge)
NAMES = ["faithfulness", "factual_correctness", "answer_relevancy", "context_precision", "context_recall"]

In [31]:
limit = asyncio.Semaphore(8)

async def score(row, record):
    contexts = [by_id[id].page_content for id in record["ids"]]
    answer = record["answer"]
    async with limit:
        results = await asyncio.gather(
            faithfulness.ascore(user_input=row.question, response=answer, retrieved_contexts=contexts),
            factual_correctness.ascore(response=answer, reference=row.reference_answer),
            answer_relevancy.ascore(user_input=row.question, response=answer),
            context_precision.ascore(user_input=row.question, reference=row.reference_answer, retrieved_contexts=contexts),
            context_recall.ascore(user_input=row.question, retrieved_contexts=contexts, reference=row.reference_answer))
    record["scores"] = {name: result.value for name, result in zip(NAMES, results)}
    record["reference"] = row.reference_answer

def rewrite_answers():
    ANSWERS.write_text("".join(json.dumps(record, ensure_ascii=False) + "\n" for record in done.values()))

for name, outs in outputs.items():
    todo = [(row, out) for row, out in zip(golden.itertuples(), outs) if out.get("reference") != row.reference_answer]
    await asyncio.gather(*(score(row, out) for row, out in todo))
    rewrite_answers()
    print(f"{name:15} scored {len(todo)}")

baseline        scored 0
+hybrid         scored 0
+self-query     scored 0
+decomposition  scored 0
+reranker       scored 0
everything      scored 0


Faithfulness and answer relevancy are averaged over questions the bot answered; context precision and recall over questions that have an answer; factual correctness over all 55 (a correct refusal scores 1):

In [32]:
def scored(outs):
    return golden.merge(pd.DataFrame([{"id": o["id"], "answer": o["answer"], **o["scores"]} for o in outs]), on="id")

def summarise(r):
    answered, has_answer = r[r.answer != REFUSAL], r[~r.should_refuse]
    return {"faithfulness": answered.faithfulness.mean(), "factual_correctness": r.factual_correctness.mean(),
            "answer_relevancy": answered.answer_relevancy.mean(),
            "context_precision": has_answer.context_precision.mean(), "context_recall": has_answer.context_recall.mean()}

results = {name: scored(outs) for name, outs in outputs.items()}
answers = pd.DataFrame({name: summarise(r) for name, r in results.items()}).T.round(2)
answers

,faithfulness,factual_correctness,answer_relevancy,context_precision,context_recall
baseline,0.93,0.64,0.77,0.59,0.79
+hybrid,0.92,0.63,0.76,0.64,0.82
+self-query,0.92,0.65,0.75,0.65,0.84
+decomposition,0.93,0.71,0.79,0.67,0.92
+reranker,0.92,0.80,0.83,0.75,0.98
everything,0.92,0.83,0.83,0.74,0.98


- **Factual correctness:** 0.64 → 0.83. **Context recall:** 0.79 → 0.98 (the answer's facts are in the chunks).
  **Context precision:** 0.59 → 0.74.
- **Faithfulness stays at 0.92–0.93 on every rung.** The bot already sticks to its sources; the fixes change what it
  reads.
- **Hybrid and self-query barely move factual correctness** (0.63, 0.65), even though they find more evidence. The
  table below shows why.

Factual correctness by question type:

In [33]:
by_type = pd.DataFrame({name: r.groupby("type").factual_correctness.mean() for name, r in results.items()}).round(2)
by_type

,baseline,+hybrid,+self-query,+decomposition,+reranker,everything
type,,,,,,
edition,0.67,0.67,0.67,0.67,0.84,0.92
fact,0.64,0.63,0.80,0.80,0.86,0.87
keyword,0.88,0.84,0.58,0.58,0.88,0.88
near-miss,0.29,0.50,0.58,0.54,0.74,0.70
paraphrase,0.93,0.67,0.72,0.56,0.83,0.89
table,0.72,0.72,0.44,0.72,0.61,0.72
two-part,0.33,0.31,0.36,0.68,0.69,0.77
unanswerable,1.00,1.00,1.00,1.00,1.00,1.00
version,0.56,0.67,0.72,0.72,0.84,0.89


Read the rows before the averages:

- **Keyword drops to 0.58 with self-query** because of one row. For k02 the bot answers `06ADRFS4076L1ZC [1]`,
  which is correct, but the judge can't split a bare code into claims and scores it 0.
- **b02 refuses with self-query.** The latest-only filter is right, but then the 2022 share must come from a table
  in the 2023 report, and search misses it.
- **Wording falls to 0.25 with everything on:** q03 again. Without the PSTN rule, the bot says SMS works and adds
  that it may not reach rural areas, but never that it's restricted.

# Part 6: Time and tokens at every step

## 17. Per question, per step

Every answer has its trace. Averaged per question: how many calls of each step, their seconds and their tokens.
The steps of one question run one after another, so a question's time is the sum of its steps.

In [34]:
traces = pd.DataFrame([{"config": name, "id": o["id"], **t} for name, outs in outputs.items() for o in outs for t in o["trace"]])
per_step = (traces.groupby(["config", "step"]).agg(calls=("seconds", "size"), seconds=("seconds", "sum"), tokens=("tokens", "sum"))
            / len(golden)).round(2)
per_step.loc["everything"].sort_values("seconds", ascending=False)

,calls,seconds,tokens
step,,,
vector,6.36,3.76,0.00
hyde,1.27,1.89,125.07
rerank,1.27,1.89,0.00
multi_query,1.27,1.66,110.55
rewrite,1.27,1.33,69.15
generate,1.00,1.20,2105.13
plan,1.27,1.11,347.80
decompose,1.00,1.06,74.55
bm25,6.36,0.01,0.00


Everything on, per question:

- **Vector search is the biggest cost: 3.8 s.** Each part searches 5 queries, and every search embeds its query
  first, over the network. BM25 is free (0.01 s).
- **Before the answer comes:** HyDE (1.9 s), the rerank (1.9 s), multi-query (1.7 s), rewrite (1.3 s), the plan
  (1.1 s) and the split (1.1 s). Each LLM step is about a second.
- **The answer itself** is 1.2 s, and the most tokens: 2,105 of the 2,832.

In [35]:
cost = traces.groupby("config").agg(seconds=("seconds", "sum"), tokens=("tokens", "sum"), llm_calls=("tokens", lambda t: (t > 0).sum()))
cost = (cost / len(golden)).reindex([c.name for c in LADDER])
cost["chunks"] = [sum(len(o["ids"]) for o in outputs[c.name]) / len(golden) for c in LADDER]
cost.round(2)

,seconds,tokens,llm_calls,chunks
config,,,,
baseline,1.58,1660.80,1.00,4.00
+hybrid,1.54,1679.98,1.00,4.00
+self-query,2.32,1956.73,2.00,4.00
+decomposition,3.78,2402.51,3.27,4.85
+reranker,5.80,2567.29,3.27,4.91
everything,13.91,2832.24,7.09,4.87


- **Baseline:** 1.6 s and 1 LLM call per question.
- **+reranker:** 5.8 s and 3.3 calls.
- **Everything on:** 13.9 s and 7.1 calls.

Tokens grow less than time (1,661 → 2,832): the steps before search are short prompts. The chunks read stay
close to 4, because only two-part questions get 8.

## 18. One question in LangSmith

`@traceable` on every step means LangSmith shows a question as a tree: `ask`, then each step under it, with the
LangChain calls inside. Tracing is off for the bulk runs; `tracing_context` turns it on for this one call. The
question is new, so every step runs live.

In [36]:
from langsmith import tracing_context

demo = "Under the 2025 NIST guideline, how long must a password be, and what does error E-530 mean on the KX-40?"
if os.getenv("LANGSMITH_API_KEY"):
    with tracing_context(enabled=True):
        out = ask(demo, EVERYTHING)
    for t in out["trace"]:
        print(f"{t['step']:11} {t['seconds']:5.2f} s {t['tokens']:5} tokens   {t['input']}")
    print("\n" + out["answer"])
save()

decompose    2.06 s   103 tokens   Under the 2025 NIST guideline, how long must a password be, and what d
plan         0.81 s   277 tokens   Under the 2025 NIST guideline, how long must a password be?
rewrite      0.78 s    57 tokens   Under the 2025 NIST guideline, how long must a password be?
multi_query  1.26 s    94 tokens   2025 NIST password minimum length requirement
hyde         1.79 s   125 tokens   2025 NIST password minimum length requirement
vector       0.50 s     0 tokens   2025 NIST password minimum length requirement
bm25         0.01 s     0 tokens   2025 NIST password minimum length requirement
vector       1.95 s     0 tokens   What is the NIST-recommended minimum password length in 2025?
bm25         0.00 s     0 tokens   What is the NIST-recommended minimum password length in 2025?
vector       0.55 s     0 tokens   What minimum password length does NIST require or recommend as of 2025
bm25         0.00 s     0 tokens   What minimum password length does NIST requir

The same steps as the trace above, as a tree in LangSmith: `ask`, then `decompose`, `plan`, `rewrite`,
`multi_query`, `hyde`, the searches, `rerank` and `generate`, each with its time, and the LangChain call inside
each LLM step. The trace above is local and free; LangSmith keeps it, so you can open any slow question later.

# Part 7: What we ship

## 19. The scorecard

In [37]:
scorecard = retrieval.join(answers).join(cost.round(2))
scorecard.to_csv("scorecard.csv")
scorecard

,hit_rate,recall,mrr,ndcg,complete,faithfulness,factual_correctness,answer_relevancy,context_precision,context_recall,seconds,tokens,llm_calls,chunks
baseline,0.84,0.77,0.69,0.67,0.70,0.93,0.64,0.77,0.59,0.79,1.58,1660.80,1.00,4.00
+hybrid,0.88,0.81,0.71,0.71,0.74,0.92,0.63,0.76,0.64,0.82,1.54,1679.98,1.00,4.00
+self-query,0.92,0.85,0.77,0.77,0.78,0.92,0.65,0.75,0.65,0.84,2.32,1956.73,2.00,4.00
+decomposition,0.92,0.92,0.76,0.79,0.92,0.93,0.71,0.79,0.67,0.92,3.78,2402.51,3.27,4.85
+reranker,0.96,0.95,0.86,0.86,0.94,0.92,0.80,0.83,0.75,0.98,5.80,2567.29,3.27,4.91
everything,0.96,0.95,0.88,0.88,0.94,0.92,0.83,0.83,0.74,0.98,13.91,2832.24,7.09,4.87


The ladder, one row per config: retrieval (ranx), answers (Ragas), and the cost per question.

## What we ship: `+reranker`

- **Hybrid search, self-query, decomposition and the bge reranker.**
- **Same retrieval as everything on:** `complete` is 0.94 for both. Factual correctness is 0.80 against 0.83.
- **Less than half the time:** 5.8 s against 13.9 s.
- **What we drop:** rewrite, multi-query and HyDE cost 8 seconds and 4 LLM calls per question for +0.03 factual
  correctness. That's within the judge's noise.

Every technique stays in the code behind its flag. Switching one on is one `replace(...)`, and the ladder shows
what it buys.

**What no flag fixes yet:** q05, a number inside a table row with no column names next to it. That's a chunking
problem, not a search problem.